<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2026</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Пространства имен. Декораторы</b><br/>
    <br/>
    <font>Ислам Абасов</font><br/>
</center>

Разберём поиск имён, замыкания и обёртки функций.


# Пространства имён (Namespaces)

Namespace связывает имена с объектами; scope определяет видимость имён.


### Четыре пространства имён в питоне

1. Built-In
2. Global
3. Enclosing
4. Local

Одинаковое имя может существовать одновременно в разных пространствах.


### Четыре пространства имён в питоне

![img](https://static-assets.codecademy.com/Courses/Intermediate-Python/Types-of-Namespaces_3_final.gif)

Вложенные области помогают понять направление поиска имени.


### Built-In (встроенный namespace)

Встроенные имена доступны без явного импорта.


### Когда создается и сколько экземпляров

Встроенное пространство существует всё время работы интерпретатора.


* Создается в момент запуска скрипта и удаляется в момент завершения скрипта
* Один на всю программу

Для обычной программы рассматриваем одно пространство встроенных имён.


### Builtins в Jupyter и обычном Python

Сравним встроенные имена текущего ядра Jupyter и отдельного процесса Python.


In [9]:
# Посмотрим имена, доступные через __builtins__ в текущем окружении.
print(dir(__builtins__))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning

In [12]:
%%python3 
# Магия IPython запускает отдельный процесс Python для сравнения окружений.
print(dir(__builtins__))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning

In [13]:
%%capture output228
%%python3 
# Сохраним вывод отдельного процесса, не показывая его на слайде.
print(dir(__builtins__))


In [14]:
# Сравним встроенные имена в ноутбуке и отдельном процессе.
set(dir(__builtins__)) - set(eval(str(output228)))  # простите за eval


{'__IPYTHON__', 'display', 'execfile', 'get_ipython', 'runfile'}

### Изменение builtins ломает чужой код

Подмена встроенного имени влияет на функции, которые находят его в builtins, включая код библиотек.
В примере непустой контейнер неожиданно становится «пустым»: проверки и ветвления начинают работать неверно.
Запускаем эксперимент в отдельном процессе и восстанавливаем исходную функцию в `finally`.


In [19]:
%%python3
# Отдельный процесс: подмена builtins не затронет ядро Jupyter.
import builtins
from collections import UserList

items = UserList([10, 20, 30])
print("До подмены:", len(items))

original_len = builtins.len
try:
    builtins.len = lambda obj: 0
    print("После подмены:", len(items))
    print("Вызов из стандартной библиотеки:", items.__len__())
    print("Непустой список считается пустым:", not items)
finally:
    builtins.len = original_len

print("После восстановления:", len(items))


До подмены: 3
После подмены: 0
Вызов из стандартной библиотеки: 0
Непустой список считается пустым: True
После восстановления: 3


<div class="alert alert-danger">
<b>Не пишите ничего в builtins!</b><br/>
Не добавляйте, не заменяйте и не удаляйте встроенные имена. Изменение общего пространства builtins может сломать ваш код, библиотеки и инструменты, которые рассчитывают на стандартное поведение Python.<br/>
Для своих имён используйте переменные, функции и модули.
</div>


### Global (глобальный namespace)

Глобальные имена принадлежат модулю, а не всем модулям сразу.


In [ ]:
# Присваивание на уровне модуля добавляет имя в globals().
XxX = 1
print(type(globals()))
assert 'XxX' in globals()


### Находим несохраненный результат из блока

IPython хранит историю ввода и результатов в пространстве ноутбука.


In [ ]:
# Результат выражения отображается без явного print().
sum([1, 2, 3])


In [ ]:
# В globals() видны и наши переменные, и служебные имена IPython.
print(globals())


In [ ]:
# Имя _i6 относится к шестому вводу IPython; это история ввода, а не результат.
_i6


In [ ]:
# del удаляет привязку имени, но не обязательно сам объект.
del _i6


### del: локальное или глобальное имя?

* `del x` удаляет привязку имени, а не обязательно сам объект.
* На уровне модуля `locals()` и `globals()` — один словарь: `del x` удаляет глобальное имя.
* В функции `del x` без `global` или `nonlocal` делает имя локальным для всего блока. Он не ищет имя по цепочке LEGB.
* После удаления локального `x` повторный `del x` вызывает `UnboundLocalError`, даже если глобальное `x` существует.
* `global x` направляет все операции с этим именем в пространство модуля; `nonlocal x` — к существующей привязке внешней функции.
* Удалить сначала локальное, затем глобальное можно явно: `del x`, затем `del globals()["x"]`. Это две разные операции; автоматического перехода нет.
* В функции нельзя надёжно удалить переменную через `del locals()["x"]`: в Python 3.13+ `locals()` возвращает снимок. В старых версиях изменение этого словаря тоже не является надёжным способом менять локальные переменные.

Демонстрация ниже использует отдельный процесс, чтобы не менять глобальные имена ноутбука.


In [ ]:
%%python3
# Локальное и глобальное x — две независимые привязки.
x = "global"

def demo():
    x = "local"
    print("До удаления:", x, globals()["x"])
    del x
    print("Глобальное имя осталось:", globals()["x"])
    try:
        del x
    except UnboundLocalError:
        print("Повторный del x не удаляет глобальное имя")
    del globals()["x"]
    print("Глобальное имя удалено:", "x" not in globals())

demo()


In [ ]:
# Чтение удалённого имени приводит к NameError.
try:
    _i6
except NameError:
    print("Variable deleted!")


### Когда создается и сколько экземпляров

Глобальное пространство связано со временем жизни модуля.


* Создается в момент запуска скрипта и удаляется в момент завершения скрипта
* У каждого модуля свой global namespace

Разные модули могут независимо определять одинаковые имена.


### Enclosing and local namespaces

Каждый вызов функции получает свои локальные имена.


In [ ]:
# Три присваивания x относятся к трём разным областям.
x = 0
def f():
    x = 1
    def g():
        # print(f"in g: {x}")  # playground, but spoiler - see later
        x = 2  # playground
        print(f"in g: {x}")
    g()
    print(f"in f: {x}")
f()
print(f"global: {x}")


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование вложенных функций
</div>

Вложенность требует внимания к тому, из какой области читается имя.


In [ ]:
# Аргументы функции тоже входят в её локальное пространство.
def f(x):
    y = 1
    print(locals())
f(10)


### Функции создают свой namespace

Локальные имена функции не становятся глобальными после вызова.


In [ ]:
# inner_variable доступна внутри функции; снаружи ожидаем NameError.
def function():
    inner_variable = 42

function()
inner_variable


In [ ]:
# locals() позволяет посмотреть локальные имена текущего вызова.
def function(arg):
    print(locals())

function(1)


In [ ]:
# Внутри функции локальное и глобальное пространства различаются.
def function(arg):
    print(locals() == globals())

function(1)


In [20]:
# На верхнем уровне ноутбука locals() и globals() обозначают одно пространство.
locals() == globals()


True

### Циклы и условия не создают свой namespace

Обычные for и if используют пространство окружающего блока.


In [ ]:
# После цикла остаются и переменная цикла, и имя из тела цикла.
k = "Hi!"

for k in range(3):
    in_for = k

print(in_for)
print(k)


In [ ]:
# Присваивание внутри if меняет имя в текущей области.
in_if = 1

if True:
    in_if = 2
 
print(in_if)


### Генераторы создают namespace

В этих примерах имя i не меняет внешнее i; детали генераторов разберём позже.


In [ ]:
# Переменная спискового включения не перезаписывает внешнее i.
i = 'Hello'
[i for i in range(10)]  # listcomp is gen
print(i)


### Variable Scope & LEGB rule

LEGB описывает обычный поиск имени в функциях.

В этой лекции разбираем поиск имён переменных, например `x`. Поиск атрибутов объектов и классов, например `obj.x` или `Class.x`, устроен сложнее и будет разобран в отдельной лекции о классах и дескрипторах.


1. Local
2. Enclosing
3. Global
4. Built-in

Поиск идёт от ближайшей области к более внешним.


### Где Python ищет имя?

![Схема поиска имени по правилу LEGB](legb_scope_resolution.png)

Источник: [Sebastian Raschka — Namespaces, Scope Resolution, and the LEGB Rule](https://www.sebastianraschka.com/Articles/2014_python_scope_and_namespaces.html).

Для обычного чтения имени в функции поиск идёт от Local к Built-in.
Если имя признано локальным, но ещё не получило значения, возникает `UnboundLocalError`: поиск наружу не продолжается.
Правила: [Python — Resolution of names](https://docs.python.org/3.14/reference/executionmodel.html#resolution-of-names).

In [21]:
# Локальное имя скрывает одноимённое глобальное; другие глобальные имена доступны.
global_var = 'global_var'
local_var = 'not_local_var'

def func(): 
    local_var = 'local_var'
    # global_var is in global namespace
    print(f'func: {global_var}')
    print(f'func: {local_var}')

func()
print(global_var)
print(local_var)


func: global_var
func: local_var
global_var
not_local_var


In [24]:
# Импорт связывает имя заново и может скрыть встроенную функцию.
print('Builtin', sum.__doc__[:50])
from numpy import sum
print('NP', sum.__doc__[:50])
from numpy import where as sum
print('NP', sum.__doc__[:50])
from builtins import sum

Builtin Return the sum of a 'start' value (default: 0) plu
NP 
Sum of array elements over a given axis.

Paramet
NP where(condition, [x, y], /)

Return elements chose


In [ ]:
# Если local_variable не определена, del тоже вызовет NameError.
del local_variable
# import gc
# gc.collect()


### Вложенные функции

Внутренняя функция может читать имена окружающей функции.


In [ ]:
# Внешняя функция не видит локальное inner_var внутренней функции.
def outer():
    outer_var = 'foo'

    def inner():
        inner_var = 'bar'
        print('outer in inner:', outer_var)
        print('inner in inner:', inner_var)

    inner()

    print('outer in outer:', outer_var)
    print('inner in outer:', inner_var)

outer()


### Замечание
Функции имеют доступ к внешним пространствам имён относительно того места где они были **определены**, а не **вызваны**

Область определения важнее места, откуда функцию вызвали.


In [ ]:
# Локальное it в q недоступно функции f; без внешнего it получим NameError.
def f():
    print(it)

def q(func):
    for it in range(10):
        func()
    print(it)

q(f)


### Когда создется объект на запись?

Присваивание имени внутри функции влияет на весь её блок.


In [32]:
# Присваивание делает global_var локальным; чтение до него даст UnboundLocalError.
global_var = 'global_var'

def func():
    # print('func0: ', global_var) # What will happen here?
    global_var = 'global_var_modified'
    print('func  :', global_var)

func()
print('global:', global_var)


func0:  global_var
func  : global_var
global: global_var


### global

global направляет присваивание в пространство текущего модуля.


In [ ]:
# Изменение global_var теперь видно после завершения вызова.
global_var = 'global_var'

def func():
    global global_var
    global_var = 'global_var_modified'
#     print(locals())
    print(global_var)

func()
print(global_var)


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование global
</div>

Глобальное состояние усложняет понимание зависимостей функции.

По общепринятому соглашению значения на уровне модуля, которые не должны меняться, оформляют как константы и называют заглавными буквами: `MAX_RETRIES`, `DEFAULT_TIMEOUT`. Python не запрещает их менять — это договорённость, а не ограничение языка.

Изменяемое рабочее состояние лучше не хранить в глобальных переменных и не добавлять напрямую в `globals()`: передавайте его через аргументы и возвращаемые значения или храните в объектах.


In [ ]:
# Изменение имени внутри функции становится побочным эффектом для вызывающего кода.
new_global_var = 'old_global_var'

def func():
    # global can create behaviour you probably don't want
    global new_global_var
    new_global_var = 'new_global_var'
    print(new_global_var)

print(new_global_var)
func()
print(new_global_var)


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование global
</div>

Неявные зависимости от глобального состояния затрудняют повторное использование.


### Проблема

global не означает ближайшую внешнюю функцию.


In [ ]:
# inner меняет var модуля, оставляя var функции outer прежним.
var = 1

def outer():
    var = 'outer'

    def inner():
        global var
        print('from global:', var)
        var = 'inner'
        print('from inner:', var)

    inner()
    print('from outer:', var)

outer()
print('from global:', var)

# cleanup
del var


### Решение: nonlocal

nonlocal выбирает существующую привязку в ближайшей внешней функции.


In [ ]:
# Теперь inner меняет var из outer, а глобальное var остаётся прежним.
var = 1

def outer():
    var = 'outer'

    def inner():
        # print('var in inner 0:', var)  # SyntaxError
        nonlocal var
        print('var in inner 1:', var)
        var = 'inner'
        print('var in inner 2:', var)

    inner()
    print('var in outer:', var)

outer()
print('var in global:', var)

del var


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование nonlocal
</div>

При изменении внешнего состояния важно понимать, кто его разделяет.


In [ ]:
# Перепривязка параметра во вложенной функции не меняет исходную переменную модуля.
# Partial reuse of enclosing variables
var = 'global value'

def foo(variable, flag=False):
    def bar():
        nonlocal variable
        if flag:
            variable = 'value'
        print('variable:', variable)
    return bar()

foo(var, flag=True)
print('var:', var)
foo(var, flag=False)
print('var:', var)


### Решение через __setattr__

Состояние можно хранить и в атрибутах объекта функции.


In [ ]:
# outer.var доступен через объект функции и сохраняется после вызова.
# without nonlocal

def outer():
    outer.var = 'v1'

    def inner():
        print('inner-0:', outer.var)
        outer.var = 'v2'
        print('inner-1:', outer.var)

    inner()
    print('outer:', outer.var)


outer()
print('global:', outer.var)


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование атрибутов функций
</div>

Атрибуты функции делают состояние доступным извне.


### Где теперь лежит переменная?

Возвращённая функция сохраняет доступ к нужным внешним привязкам.


In [ ]:
# inner читает enclosing_var из outer, даже когда вызов outer уже завершился.
def outer():
    enclosing_var = 'outer'

    def inner():
        print('from inner :', enclosing_var)

    return inner

enclosing_var = 'global'
outer()()
print('from global:', enclosing_var)


## Замыкания [Closures]

**Замыкание** — функция, которая сохраняет доступ к переменным внешней функции даже после того, как вызов внешней функции завершился.


Существования замыканий следует из правила LEGB и возможностью оперировать с функциями как обьектами.

Функцию можно вернуть как значение, сохранив её внешние привязки.


In [ ]:
# Каждая созданная функция запоминает свой multiplier; типизация здесь предварительная.
from typing import Callable, TypeVar
_T = TypeVar(name='_T')  # spoilers

def make_multiplicator(multiplier: int) -> Callable[[_T], _T]:
    def _multiplicator(data: _T) -> _T:
        return data * multiplier
    return _multiplicator


In [ ]:
# Два независимых замыкания умножают строки на разные значения.
doubler = make_multiplicator(2)
tripler = make_multiplicator(3)

doubler('A') + tripler('B')


In [ ]:
# Создание замыкания и его вызов можно записать в одном выражении.
make_multiplicator(2)(7)


In [ ]:
# Замыкание хранит ссылку на coefs; изменение этого списка повлияет на вычисления.
from typing import Callable

def make_polynom(coefs: list[float]) -> Callable[[float], float]:
    def _polynom(x: float) -> float:
        return sum(coef * x ** idx for idx, coef in enumerate(coefs))
    return _polynom

coef = [0, 0, 1]
_pow2 = make_polynom(coef)
print(_pow2(5))

In [35]:
# Что будет?
coef[0] = -5
print(_pow2(5))

25
20


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование сложных замыканий
</div>

Чем больше скрытого состояния, тем сложнее отслеживать поведение замыкания.


### Работаем с функциями как с классами

Несколько функций могут разделять одну внешнюю переменную.


In [36]:
# Getter и setter одной фабрики разделяют value; разные вызовы фабрики независимы.
def value_factory(value=10):
    def _value_getter():
        return value

    def _value_setter(new_value):
        nonlocal value
        value = new_value
        return value

    return _value_getter, _value_setter

get_value, set_value = value_factory(100)
get_value2, set_value2 = value_factory(100)
print(get_value(), get_value2())

set_value(10**6)

print(get_value(), get_value2())


100 100
1000000 100


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование замыканий вместо классов
</div>

При большом числе операций со состоянием стоит оценить удобство класса.


In [ ]:
# Метод update здесь тоже функция и меняет захваченное x.
def make_adder(x):
    def add(y):
        return x + y

    def zxc(new_x):
        nonlocal x
        x = new_x

    add.update = zxc
    return add


adder = make_adder(10)
print(adder(5))

adder.update(100)
print(adder(23))
print(adder(45))


<div class="alert alert-danger">
<b>Антипаттерн: </b> использование замыканий вместо классов
</div>

Такой интерфейс требует понимания общего состояния двух функций.



# Декораторы

Декоратор принимает объект и возвращает объект, который заменит исходный.


In [37]:
# Первый пример применяет декоратор обычным вызовом функции.
import sys

def deprecated(func):
    def inner(*args, **kwargs):
        print(f'{func.__name__} is deprecated!', file=sys.stderr)
        return func(*args, **kwargs)
    return inner

depr_print = deprecated(print)
depr_print([1, 2, 3])


[1, 2, 3]


print is deprecated!


### Синтаксис декораторов

Запись с @ связывает имя с результатом декорирования.


In [ ]:
# Обёртка передаёт исходной функции аргументы и возвращает её результат.
import sys

def deprecated(func):
    def wrapper(*args, **kwargs):
        print(f'{func.__name__} is deprecated!', file=sys.stderr)
#         return smth
        return func(*args, **kwargs)
    return wrapper

@deprecated
def add(x, y):
    return x + y

add(1, 2)


### Когда применяется декоратор?

`@deprecated` перед `def add(...)` соответствует `add = deprecated(add)` после определения.

Декоратор вызывается при выполнении определения функции, обычно при импорте модуля.
Тело `wrapper` выполняется позже — при каждом вызове декорированной функции.

### Monke

Результат декоратора не обязан быть функцией.


In [39]:
# Имя dummy_function теперь связано с HTML-объектом.
from IPython import display

def monke(func):
    return display.HTML('<img src="https://i.kym-cdn.com/photos/images/original/002/578/699/9c8.gif">')

@monke
def dummy_function(arg):
    print(arg)

# dummy_function = monke(dummy_function)

print(type(dummy_function))
dummy_function  # not a function anymore


<class 'IPython.core.display.HTML'>


### Проблема

Без копирования метаданных мы видим свойства обёртки.


In [44]:
# Имя и docstring show потерялись, потому что show теперь указывает на wrapper.
@deprecated
def show(x):
    'This is a really nice looking docstring'
    print(x)

print(show.__name__)
print(show.__doc__)


inner
None


In [45]:
# Вызов без x даст TypeError; текст ошибки относится к исходной функции.
show()

show is deprecated!


TypeError: show() missing 1 required positional argument: 'x'

### Решение 1

Метаданные можно переносить вручную, но легко забыть часть атрибутов.


In [46]:
# Ручное копирование восстанавливает только выбранные свойства.
def deprecated(func):
    def wrapper(*args, **kwargs):
        print('{} is deprecated!'.format(func.__name__),
              file=sys.stderr)
        return func(*args, **kwargs)
    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__module__ = func.__module__
    return wrapper

@deprecated
def show(x):
    'This is a really nice looking docstring'
    print(x)

print(show.__name__)
print(show.__doc__)


show
This is a really nice looking docstring


In [47]:
# Даже после копирования метаданных обязательный аргумент x остаётся обязательным.
show()


show is deprecated!


TypeError: show() missing 1 required positional argument: 'x'

### Решение 2

Стандартный wraps переносит метаданные и добавляет ссылку __wrapped__.


In [ ]:
# Теперь инструменты могут найти исходную функцию через цепочку обёрток.
from functools import wraps

def deprecated(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('{} is deprecated!'.format(func.__name__),
              file=sys.stderr)
        return func(*args, **kwargs)
    return wrapper

@deprecated
def show(x):
    'This is a really nice looking docstring'
    print(x)

print(show.__name__)
print(show.__doc__)


### Декоратор Once

Состояние декоратора принадлежит конкретной декорированной функции.


**Поведение once**

Первый успешный вызов сохраняет результат, последующие возвращают его даже с другими аргументами.
Если вызов завершился исключением, следующая попытка снова вызывает исходную функцию.
Это учебный вариант для последовательных нерекурсивных вызовов: защиты от параллельного выполнения нет.

In [50]:
# Один успешный вызов сохраняет результат; исключение допускает новую попытку.
from functools import wraps

def once(func):
    called = False
    result = None

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal called, result
        if not called:
            result = func(*args, **kwargs)
            called = True  # Только после успешного завершения.
        return result

    return wrapper

@once
def f():
    print('Hi!')
    return 42

# f = once(f)

@once
def g():
    print('Hello!')
    return 7

print(f(), f())
print(g(), g())


Hi!
42 42
Hello!
7 7


### Декоратор с параметрами

В `trace` три уровня функций:

1. `trace(destination)` получает настройки и возвращает декоратор.
2. `decorate_function(original_function)` получает исходную функцию и возвращает обёртку.
3. `traced_function(*args, **kwargs)` выполняется при каждом вызове: пишет сообщение и вызывает исходную функцию.

`@trace(sys.stdout)` соответствует `add = trace(sys.stdout)(add)`.
Первые два уровня выполняются при определении `add`, третий — при вызове `add(2, 3)`.


In [ ]:
# Фабрика → декоратор → обёртка: каждому уровню нужны свои аргументы.
from functools import wraps
import sys


def trace(destination=sys.stderr):
    # 1. Запоминаем, куда выводить сообщения.
    def decorate_function(original_function):
        # 2. Запоминаем, какую функцию нужно обернуть.
        @wraps(original_function)
        def traced_function(*args, **kwargs):
            # 3. При каждом вызове выводим сообщение и передаём аргументы дальше.
            print(
                f'{original_function.__name__} called with args {args}, {kwargs}!',
                file=destination,
            )
            return original_function(*args, **kwargs)

        return traced_function  # Возвращаем обёртку, а не вызываем её.

    return decorate_function  # Возвращаем настроенный декоратор.


@trace(sys.stdout)
def add(left, right):
    return left + right


print(add(2, 3))  # Сообщение о вызове, затем результат 5.


### Цепочки декораторов

Несколько декораторов образуют вложенные обёртки.


In [ ]:
# Сначала создаётся trace-обёртка, затем deprecated-обёртка вокруг неё.
@deprecated
@trace()
def f(x):
    return x

f(1)


### Порядок декораторов

```python
@deprecated
@trace()
def f(x):
    return x
```

Соответствует `f = deprecated(trace()(f))`: обёртки применяются снизу вверх.
При вызове сначала работает внешняя `deprecated`, затем `trace`, затем исходная функция.
Выражения после `@` вычисляются сверху вниз при выполнении определения.

## Декоратор notify

Уведомление можно показать без внешних сервисов.


In [ ]:
# После успешного вызова печатаем сообщение и возвращаем исходный результат.
from functools import wraps

def notify(*, message="Готово"):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            print(message)
            return result
        return wrapper
    return decorator

@notify()
def foo():
    return 42

@notify(message="Сумма посчитана")
def add(x, y):
    return x + y

print(foo())
print(add(2, 3))


### Один интерфейс для notify

Всегда используем скобки: `@notify()` или `@notify(message="…")`.
Фабрика получает настройки, декоратор — функцию, обёртка — аргументы вызова.
Если исходная функция выбросит исключение, сообщение не печатается.

Для параметров используем явную фабрику декоратора с единым интерфейсом.


# Тайпинги

[PEP 612](https://peps.python.org/pep-0612/): `ParamSpec` и `Concatenate`.

* `P = ParamSpec("P")` сохраняет набор параметров исходной функции.
* `R = TypeVar("R")` связывает тип её результата с результатом обёртки.
* `Callable[P, R]` — функция с параметрами `P` и результатом `R`.
* `P.args` и `P.kwargs` описывают передаваемые аргументы.
* `Concatenate` нужен, если декоратор добавляет параметр; здесь параметры не меняются.

Типизация помогает сохранить параметры и результат декорируемой функции.


In [ ]:
# P описывает параметры, R — результат; аннотации не проверяют значения при выполнении.
from functools import wraps
from typing import Callable, TypeVar, TextIO, ParamSpec
import sys

R = TypeVar("R")
P = ParamSpec("P")

def trace(destination: TextIO = sys.stderr) -> Callable[[Callable[P, R]], Callable[P, R]]:
    def decorate_function(original_function: Callable[P, R]) -> Callable[P, R]:
        @wraps(original_function)
        def traced_function(*args: P.args, **kwargs: P.kwargs) -> R:
            print(
                f'{original_function.__name__} called with args {args}, {kwargs}!',
                file=destination,
            )
            return original_function(*args, **kwargs)
        return traced_function
    return decorate_function


# Изменения Python, важные для этой лекции

Главные изменения: когда вычисляются аннотации (3.14), что возвращает `locals()` внутри функции (3.13), и стандартный декоратор устаревания (3.13).

Далее сравним старое и новое поведение. Версии указаны у каждого примера; это материал для чтения, а не единая последовательность команд.
Синтаксис `@decorator` в 3.14 не изменился.


### Python 3.14: аннотации вычисляются позже

**Раньше, до 3.14:** без `from __future__ import annotations` выражения в аннотациях вычислялись при выполнении `def`.

```python
def choose_type():
    print("Вычисляем аннотацию")
    return int

def double(x: choose_type()):
    return x * 2

print("Функция определена")
# До 3.14: сначала «Вычисляем аннотацию», затем «Функция определена».
# В 3.14: пока только «Функция определена».
print(double.__annotations__)  # {'x': <class 'int'>}
# В 3.14 именно здесь впервые вычисляется аннотация.
```

Обычный вызов `double(3)` не вычисляет аннотацию и не проверяет тип аргумента.
**Для декораторов:** чтение `func.__annotations__` может запустить вычисление. Для переноса метаданных используем `functools.wraps`, а не ручное копирование.

[Документация Python 3.14](https://docs.python.org/3.14/whatsnew/3.14.html#deferred-evaluation-of-annotations).


### Python 3.14: имя в аннотации можно определить позже

**Раньше:** если `Result` ещё не определён, `def build() -> Result` давал `NameError`. Использовали строку:

```python
def build() -> "Result":
    return Result()

class Result:
    pass
```

**Теперь, в 3.14:** кавычки обычно не нужны — имя должно существовать к моменту вычисления аннотации.

```python
def build() -> Result:
    return Result()

class Result:
    pass

print(build.__annotations__["return"] is Result)  # True
```

Если читать аннотации до определения `Result`, вычисление значения всё ещё даст `NameError`.
Для строкового представления есть новый модуль:

```python
from annotationlib import Format, get_annotations
print(get_annotations(build, format=Format.STRING))  # {'return': 'Result'}
```

`from __future__ import annotations` — отдельный режим: сохраняет аннотации строками; он не тождествен новому поведению по умолчанию.
[Как читать аннотации](https://docs.python.org/3.14/howto/annotations.html).


### Python 3.13: locals() — снимок внутри функции

**Раньше, в CPython до 3.13:** повторные вызовы `locals()` в функции возвращали один словарь и обновляли его. Сохранённый словарь мог неожиданно поменяться.
**Теперь:** каждый вызов возвращает новый снимок.

```python
def demo():
    x = 1
    before = locals()
    x = 2
    after = locals()
    print(before is after)
    print(before["x"], after["x"])

demo()
# CPython 3.12: True; 2 2
# Python 3.13+: False; 1 2
```

Это сравнение конкретного старого поведения CPython: ранее нельзя было полагаться на единую семантику изменения этого словаря.

```python
def demo():
    x = 1
    snapshot = locals()
    snapshot["x"] = 99
    print(x)  # В 3.13+ гарантированно 1: изменён снимок, а не переменная.
```

На уровне модуля `locals()` и `globals()` остаются одним пространством. Внутри функции для изменения переменной используем присваивание, а не словарь `locals()`.
[Документация Python 3.13](https://docs.python.org/3.13/whatsnew/3.13.html#defined-mutation-semantics-for-locals).


### Python 3.13: стандартный deprecated

**Раньше:** писали собственную обёртку. Наш учебный вариант печатает в `stderr`; для настоящего предупреждения использовали `warnings.warn`:

```python
from functools import wraps
import warnings

def deprecated(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        warnings.warn("Используйте new_sum", DeprecationWarning, stacklevel=2)
        return func(*args, **kwargs)
    return wrapper
```

**Теперь:** готовый декоратор есть в стандартной библиотеке.

```python
from warnings import deprecated

@deprecated("Используйте new_sum")
def old_sum(a, b):
    return a + b
```

Результат вызова сохраняется. При вызове выдаётся `DeprecationWarning`; статический анализатор с поддержкой PEP 702 может подсветить использование устаревшей функции.
Предупреждения могут скрываться фильтрами, поэтому для демонстрации:

```python
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("always", DeprecationWarning)
    print(old_sum(2, 3))  # Предупреждение и результат 5.
```

[Документация warnings.deprecated](https://docs.python.org/3.13/library/warnings.html#warnings.deprecated).


### Python 3.14: partial может оставить «место» для аргумента

`partial` создаёт функцию с частью заранее заданных аргументов.
У `str.replace` аргументы `old` и `new` передаются только позиционно: зафиксировать их по имени нельзя.

**Раньше:** чтобы заранее выбрать замену, оставив строку аргументом будущего вызова, писали обёртку.

```python
replace_spaces = lambda text: str.replace(text, " ", "_")
print(replace_spaces("hello world"))  # hello_world
```

**Теперь, в 3.14:** `Placeholder` оставляет место для строки перед уже заданными аргументами.

```python
from functools import partial, Placeholder
replace_spaces = partial(str.replace, Placeholder, " ", "_")
print(replace_spaces("hello world"))  # hello_world
```

Все оставленные места нужно заполнить при вызове. Раньше `partial` позволял фиксировать позиционно только начальные аргументы.
Именованные аргументы можно было фиксировать и раньше: например, `partial(pow, exp=2)` работает до Python 3.14.
[Документация functools.Placeholder](https://docs.python.org/3.14/library/functools.html#functools.Placeholder).


### Python 3.12: короче запись типов декоратора

**Раньше:** параметры типов объявляли отдельно; запись по-прежнему работает.

```python
from typing import Callable, ParamSpec, TypeVar
P = ParamSpec("P")
R = TypeVar("R")

def decorator(func: Callable[P, R]) -> Callable[P, R]:
    return func  # Минимальный пример: параметры и результат сохраняются.
```

**Теперь:** параметры типов можно объявить рядом с именем функции.

```python
from typing import Callable

def decorator[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    return func
```

`**P` описывает параметры, `R` — результат. Меняется запись типов, а не порядок работы декоратора.
С 3.12 `functools.wraps` сохраняет и `__type_params__`.
[Синтаксис параметров типов](https://docs.python.org/3.12/whatsnew/3.12.html#pep-695-type-parameter-syntax), [wraps](https://docs.python.org/3.14/library/functools.html#functools.update_wrapper).


### Python 3.12: списковые включения выполняются иначе

**Раньше, в CPython до 3.12:** для спискового включения создавалась скрытая функция с отдельным кадром выполнения.
**Теперь:** включение встроено в окружающий код. Имя переменной цикла всё равно изолировано.

```python
i = "внешнее значение"
values = [i * 2 for i in range(3)]
print(values)  # [0, 2, 4]
print(i)       # «внешнее значение» и раньше, и сейчас.
```

Обычно результат прежний. Разницу видно, например, при просмотре локальных имён внутри включения:

```python
def demo():
    marker = 42
    return ["marker" in locals() for i in range(1)]

print(demo())
# CPython 3.11: [False] — locals() скрытой функции.
# CPython 3.12+: [True] — locals() окружающей функции.
```

Списковое включение создаёт список, а не генератор; внутренняя реализация не меняет этого различия.
[PEP 709 в документации Python 3.12](https://docs.python.org/3.12/whatsnew/3.12.html#pep-709-comprehension-inlining).
